In [1]:
import os
import json
from pathlib import Path

def generate_local_metadata(data_root_dir, output_jsonl_path):
    """
    Scans local data folders and creates a unified metadata file for training.
    Works perfectly across Windows, Mac, and Linux systems.
    """
    print(f"🔍 Scanning data root directory: {data_root_dir}")
    data_list = []

    # Supported image extensions
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}

    # Counter for tracking
    stats = {'train': 0, 'val': 0, 'test': 0}

    # Recursively traverse through all folders
    for root, _, files in os.walk(data_root_dir):
        for filename in files:
            ext = os.path.splitext(filename)[1].lower()
            if ext in valid_extensions:
                full_path = os.path.join(root, filename)

                # Convert path to pure pathlib object to isolate platform-safe parts
                path_obj = Path(full_path)
                parts = path_obj.parts

                # We expect the path to look like: .../data/split/label/filename.jpg
                # parts[-3] -> split (train/val/test)
                # parts[-2] -> label (fractured/not fractured)
                try:
                    split = parts[-3].lower()
                    label = parts[-2].lower()

                    # Validate that the folder name belongs to expected partitions
                    if split in ['train', 'val', 'test']:

                        # Determine clinical ground truth
                        is_fractured = "not" not in label

                        question = "Is there a bone fracture in this X-ray image?"
                        answer = "Yes, there is a fracture." if is_fractured else "No, there is no fracture."

                        # Added the curable flag exactly as you requested
                        curable = "yes" if is_fractured else "no"

                        data_list.append({
                            "image": os.path.abspath(full_path),
                            "question": question,
                            "answer": answer,
                            "curable": curable,
                            "split": split
                        })
                        stats[split] += 1
                except IndexError:
                    # Skips any files that aren't deeply nested enough to avoid crashes
                    continue

    # Write dynamically to a JSON Lines file
    with open(output_jsonl_path, 'w', encoding='utf-8') as f:
        for entry in data_list:
            json.dump(entry, f, ensure_ascii=False)
            f.write('\n')

    print("\n" + "="*50)
    print("✨ LOCAL METADATA GENERATION SUCCESSFUL ✨")
    print("="*50)
    print(f"📂 Saved File -> {os.path.abspath(output_jsonl_path)}")
    print(f"📊 Train Images Processed : {stats['train']}")
    print(f"📊 Val Images Processed   : {stats['val']}")
    print(f"📊 Test Images Processed  : {stats['test']}")
    print(f"📈 Total Tracked Records  : {len(data_list)}")
    print("="*50)

if __name__ == "__main__":
    # Your exact Mac absolute path
    DATA_ROOT = "/Users/saimohith/Documents/Sem-5/DL/BoneFractureQA/data"

    # Saves the JSONL directly into your BoneFractureQA project folder
    OUTPUT_FILE = "/Users/saimohith/Documents/Sem-5/DL/BoneFractureQA/metadata.jsonl"

    generate_local_metadata(DATA_ROOT, OUTPUT_FILE)

🔍 Scanning data root directory: /Users/saimohith/Documents/Sem-5/DL/BoneFractureQA/data

✨ LOCAL METADATA GENERATION SUCCESSFUL ✨
📂 Saved File -> /Users/saimohith/Documents/Sem-5/DL/BoneFractureQA/metadata.jsonl
📊 Train Images Processed : 9246
📊 Val Images Processed   : 829
📊 Test Images Processed  : 506
📈 Total Tracked Records  : 10581
